# Cyclistic Q1 2019 2020  

Initial exploration with pandas. This step revealed that the average ride_length is distorted by outliers, so the final analysis was done in SQL using medians (see /Queries). The original datasets can be downloaded from [Divvy](https://divvy-tripdata.s3.amazonaws.com/index.html).

In [89]:
import pandas as pd

In [ ]:
# Upload Divvy datasets (csv files)
q1_2019 = pd.read_csv("CSVs/Datasets/Cyclistic_Trips_2019_Q1.csv")
q1_2020 = pd.read_csv("CSVs/Datasets/Cyclistic_Trips_2020_Q1.csv")

## WRANGLE DATA AND COMBINE INTO A SINGLE FILE

In [91]:
# Compare column names for each of the files
print("Columns for Q1 2019:\n", q1_2019.columns)
print("\nColumns for Q1 2020:\n", q1_2020.columns)

Columns for Q1 2019:
 Index(['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration',
       'from_station_id', 'from_station_name', 'to_station_id',
       'to_station_name', 'usertype', 'gender', 'birthyear'],
      dtype='object')

Columns for Q1 2020:
 Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')


In [92]:
# Rename columns to make them consistent with q1_2020
q1_2019 = q1_2019.rename(columns={
    'trip_id': 'ride_id',
    'bikeid': 'rideable_type',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'from_station_name': 'start_station_name',
    'from_station_id': 'start_station_id',
    'to_station_name': 'end_station_name',
    'to_station_id': 'end_station_id',
    'usertype': 'member_casual'
})

In [93]:
# Inspect the data frames and look for incongruities
print("\nQ1 2019 Data Info:")
q1_2019.info()
print("\nQ1 2020 Data Info:")
q1_2020.info()


Q1 2019 Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365069 entries, 0 to 365068
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             365069 non-null  int64  
 1   started_at          365069 non-null  object 
 2   ended_at            365069 non-null  object 
 3   rideable_type       365069 non-null  int64  
 4   tripduration        365069 non-null  object 
 5   start_station_id    365069 non-null  int64  
 6   start_station_name  365069 non-null  object 
 7   end_station_id      365069 non-null  int64  
 8   end_station_name    365069 non-null  object 
 9   member_casual       365069 non-null  object 
 10  gender              345358 non-null  object 
 11  birthyear           347046 non-null  float64
dtypes: float64(1), int64(4), object(7)
memory usage: 33.4+ MB

Q1 2020 Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426887 entries, 0 to 426886
Data col

In [94]:
# Convert ride_id and rideable_type to string so that they can stack correctly
q1_2019['ride_id'] = q1_2019['ride_id'].astype(str)
q1_2019['rideable_type'] = q1_2019['rideable_type'].astype(str)

In [95]:
# Stack individual quarter's data frames into one big data frame
all_trips = pd.concat([q1_2019, q1_2020], ignore_index=True)

In [ ]:
# Remove lat, long, birthyear, and gender fields as this data was dropped beginning in 2020
all_trips = all_trips.drop(columns=['start_lat', 'start_lng', 'end_lat', 'end_lng', 'birthyear', 'gender', 'tripduration'], errors='ignore')

## CLEAN UP AND ADD DATA TO PREPARE FOR ANALYSIS

In [97]:
# Inspect the new table that has been created
print("\nList of column names:\n", all_trips.columns)
print("\nHow many rows are in data frame?", len(all_trips))
print("\nDimensions of the data frame:", all_trips.shape)
print("\nSee the first 6 rows of data frame:\n", all_trips.head())
print("\nSee list of columns and data types:\n")
all_trips.info()


List of column names:
 Index(['ride_id', 'started_at', 'ended_at', 'rideable_type',
       'start_station_id', 'start_station_name', 'end_station_id',
       'end_station_name', 'member_casual'],
      dtype='object')

How many rows are in data frame? 791956

Dimensions of the data frame: (791956, 9)

See the first 6 rows of data frame:
     ride_id           started_at             ended_at rideable_type  \
0  21742443  2019-01-01 00:04:37  2019-01-01 00:11:07          2167   
1  21742444  2019-01-01 00:08:13  2019-01-01 00:15:34          4386   
2  21742445  2019-01-01 00:13:23  2019-01-01 00:27:12          1524   
3  21742446  2019-01-01 00:13:45  2019-01-01 00:43:28           252   
4  21742447  2019-01-01 00:14:52  2019-01-01 00:20:56          1170   

   start_station_id                   start_station_name  end_station_id  \
0               199               Wabash Ave & Grand Ave            84.0   
1                44               State St & Randolph St           624.0   
2   

In [ ]:
# Statistical summary of data
print("\nStatistical summary of data:\n", all_trips.describe())


Statistical summary of data:
        start_station_id  end_station_id
count     791956.000000   791955.000000
mean         204.400294      204.379354
std          158.919132      159.323552
min            2.000000        2.000000
25%           77.000000       77.000000
50%          174.000000      174.000000
75%          291.000000      291.000000
max          675.000000      675.000000


In [ ]:
# In the "member_casual" column,  "Subscriber" in replaced with "member" and "Customer" with "casual"
print("\nValue counts for 'member_casual' column before cleaning:\n", all_trips['member_casual'].value_counts())


Value counts for 'member_casual' column before cleaning:
 member_casual
member        378407
Subscriber    341906
casual         48480
Customer       23163
Name: count, dtype: int64


In [ ]:
# Reassign to the current 2020 labels
all_trips['member_casual'] = all_trips['member_casual'].replace({
    'Subscriber': 'member',
    'Customer': 'casual'
})

In [101]:
# Check to make sure the proper number of observations were reassigned
print("\nValue counts for 'member_casual' column after cleaning:\n", all_trips['member_casual'].value_counts())


Value counts for 'member_casual' column after cleaning:
 member_casual
member    720313
casual     71643
Name: count, dtype: int64


In [102]:
# Convert 'started_at' and 'ended_at' to datetime objects
all_trips['started_at'] = pd.to_datetime(all_trips['started_at'])
all_trips['ended_at'] = pd.to_datetime(all_trips['ended_at'])

In [103]:
# Add columns that list the date, month, day, and year of each ride
all_trips['date'] = all_trips['started_at'].dt.date
all_trips['month'] = all_trips['started_at'].dt.month_name()
all_trips['day'] = all_trips['started_at'].dt.day
all_trips['year'] = all_trips['started_at'].dt.year
all_trips['day_of_week'] = all_trips['started_at'].dt.day_name()

In [104]:
# Add a "ride_length" calculation to all_trips (in seconds)
all_trips['ride_length'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()

In [105]:
# Inspect the structure of the columns
print("\nData types after adding date columns and ride_length:\n")
all_trips.info()


Data types after adding date columns and ride_length:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 791956 entries, 0 to 791955
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             791956 non-null  object        
 1   started_at          791956 non-null  datetime64[ns]
 2   ended_at            791956 non-null  datetime64[ns]
 3   rideable_type       791956 non-null  object        
 4   start_station_id    791956 non-null  int64         
 5   start_station_name  791956 non-null  object        
 6   end_station_id      791955 non-null  float64       
 7   end_station_name    791955 non-null  object        
 8   member_casual       791956 non-null  object        
 9   date                791956 non-null  object        
 10  month               791956 non-null  object        
 11  day                 791956 non-null  int32         
 12  year                791956 non

In [ ]:
# Remove "bad" data
# The data frame includes entries where bikes were taken out of docks for quality checks or ride_length was negative
all_trips_v2 = all_trips[(all_trips['start_station_name'] != "HQ QR") & (all_trips['ride_length'] >= 0)].copy()

## DESCRIPTIVE ANALYSIS

In [107]:
# Descriptive analysis on ride_length (all figures in seconds)
print("\nDescriptive statistics for ride_length:\n", all_trips_v2['ride_length'].describe())


Descriptive statistics for ride_length:
 count    7.881890e+05
mean     1.189459e+03
std      3.329191e+04
min      1.000000e+00
25%      3.310000e+02
50%      5.390000e+02
75%      9.120000e+02
max      1.063202e+07
Name: ride_length, dtype: float64


In [108]:
# Descriptive analysis on ride_length (all figures in minutes)
minutes = all_trips_v2['ride_length'] / 60
print("\nDescriptive statistics for ride_length in minutes:\n", minutes.describe())


Descriptive statistics for ride_length in minutes:
 count    788189.000000
mean         19.824311
std         554.865218
min           0.016667
25%           5.516667
50%           8.983333
75%          15.200000
max      177200.366667
Name: ride_length, dtype: float64


In [109]:
# Descriptive analysis on ride_length of members (all figures in minutes)
minutes_members = all_trips_v2[all_trips_v2['member_casual'] == 'member']['ride_length'] / 60
print("\nDescriptive statistics for ride_length of members in minutes:\n", minutes_members.describe())


Descriptive statistics for ride_length of members in minutes:
 count    720312.000000
mean         13.254206
std         273.236695
min           0.016667
25%           5.283333
50%           8.466667
75%          13.716667
max      101607.133333
Name: ride_length, dtype: float64


In [110]:
# Descriptive analysis on ride_length of casual users (all figures in minutes)
minutes_casual = all_trips_v2[all_trips_v2['member_casual'] == 'casual']['ride_length'] / 60
print("\nDescriptive statistics for ride_length of casual users in minutes:\n", minutes_casual.describe())


Descriptive statistics for ride_length of casual users in minutes:
 count     67877.000000
mean         89.546398
std        1666.582759
min           0.033333
25%          13.000000
50%          23.216667
75%          39.316667
max      177200.366667
Name: ride_length, dtype: float64


In [111]:
# Compare members and casual users
print("\nRide length statistics grouped by member_casual:\n", all_trips_v2.groupby('member_casual')['ride_length'].agg(['mean', 'median', 'max', 'min']))


Ride length statistics grouped by member_casual:
                       mean  median         max  min
member_casual                                      
casual         5372.783874  1393.0  10632022.0  2.0
member          795.252339   508.0   6096428.0  1.0


In [112]:
# See the average ride time by each day for members vs casual users
print("\nAverage ride length by member type and day of the week:\n", all_trips_v2.groupby(['member_casual', 'day_of_week'])['ride_length'].mean())


Average ride length by member type and day of the week:
 member_casual  day_of_week
casual         Friday         6090.737302
               Monday         4752.050438
               Saturday       4950.770801
               Sunday         5061.304364
               Thursday       8451.666853
               Tuesday        4561.803857
               Wednesday      4480.372432
member         Friday          796.733789
               Monday          822.311202
               Saturday        974.072964
               Sunday          972.938336
               Thursday        707.209274
               Tuesday         769.441629
               Wednesday       711.983790
Name: ride_length, dtype: float64


In [113]:
# Order the days of the week for correct sorting for analysis
days_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
all_trips_v2['day_of_week'] = pd.Categorical(all_trips_v2['day_of_week'], categories=days_order, ordered=True)


In [114]:
# Order the months of the year for correct sorting for analysis
months_order = ["January", "February", "March"]
all_trips_v2['month'] = pd.Categorical(all_trips_v2['month'], categories=months_order, ordered=True)


In [ ]:
print("\nAverage ride length (sorted by day of week):\n", all_trips_v2.groupby(['member_casual', 'day_of_week'])['ride_length'].mean())


Average ride length (sorted by day of week):
 member_casual  day_of_week
casual         Sunday         5061.304364
               Monday         4752.050438
               Tuesday        4561.803857
               Wednesday      4480.372432
               Thursday       8451.666853
               Friday         6090.737302
               Saturday       4950.770801
member         Sunday          972.938336
               Monday          822.311202
               Tuesday         769.441629
               Wednesday       711.983790
               Thursday        707.209274
               Friday          796.733789
               Saturday        974.072964
Name: ride_length, dtype: float64


/var/folders/0m/74b0vn150zdbflk7dmsw0t140000gp/T/ipykernel_46928/192120281.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print("\nAverage ride length (sorted by day of week):\n", all_trips_v2.groupby(['member_casual', 'day_of_week'])['ride_length'].mean())


In [116]:
# Analyze ridership data by type and weekday
summary_stats = all_trips_v2.groupby(['member_casual', 'day_of_week']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_length', 'mean')
).reset_index()

print("\nSummary of rides and duration by rider type and weekday:\n", summary_stats)


Summary of rides and duration by rider type and weekday:
    member_casual day_of_week  number_of_rides  average_duration
0         casual      Sunday            18652       5061.304364
1         casual      Monday             5591       4752.050438
2         casual     Tuesday             7311       4561.803857
3         casual   Wednesday             7690       4480.372432
4         casual    Thursday             7147       8451.666853
5         casual      Friday             8013       6090.737302
6         casual    Saturday            13473       4950.770801
7         member      Sunday            60197        972.938336
8         member      Monday           110430        822.311202
9         member     Tuesday           127974        769.441629
10        member   Wednesday           121902        711.983790
11        member    Thursday           125228        707.209274
12        member      Friday           115168        796.733789
13        member    Saturday            59413

/var/folders/0m/74b0vn150zdbflk7dmsw0t140000gp/T/ipykernel_46928/3581701072.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_stats = all_trips_v2.groupby(['member_casual', 'day_of_week']).agg(


In [117]:
# Analyze ridership data by type and month
summary_stats_2 = all_trips_v2.groupby(['member_casual', 'month']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_length', 'mean')
).reset_index()

print("\nSummary of rides and duration by rider type and month:\n", summary_stats_2)


Summary of rides and duration by rider type and month:
   member_casual     month  number_of_rides  average_duration
0        casual   January            12387       7150.543150
1        casual  February            14952       8127.569623
2        casual     March            40538       3813.489861
3        member   January           234769        782.444688
4        member  February           220262        783.610432
5        member     March           265281        816.253124


/var/folders/0m/74b0vn150zdbflk7dmsw0t140000gp/T/ipykernel_46928/1936233943.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_stats_2 = all_trips_v2.groupby(['member_casual', 'month']).agg(


## EXPORT SUMMARY FILE FOR FURTHER ANALYSIS

In [118]:
# Create a .csv file to visualize elsewhere
counts = all_trips_v2.groupby(['member_casual', 'day_of_week', 'month'])['ride_length'].mean().reset_index()
counts

/var/folders/0m/74b0vn150zdbflk7dmsw0t140000gp/T/ipykernel_46928/1945822977.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = all_trips_v2.groupby(['member_casual', 'day_of_week', 'month'])['ride_length'].mean().reset_index()


,member_casual,day_of_week,month,ride_length
0,casual,Sunday,January,8570.871400
1,casual,Sunday,February,7175.534929
2,casual,Sunday,March,3709.068715
3,casual,Monday,January,9951.989121
4,casual,Monday,February,4430.635116
5,casual,Monday,March,3047.574420
6,casual,Tuesday,January,3748.319135
7,casual,Tuesday,February,10341.775544
8,casual,Tuesday,March,3324.639525
9,casual,Wednesday,January,8989.857056


In [119]:
counts.to_csv('avg_ride_length.csv', index=False)